# מפגש 14 — מחברת הדגמה מקצועית
Train מלמד שמות קטגוריות ואת פרמטרי ההכנה. Test/New משתמשת באותה סכמה, **ללא Fit חדש**. זו הכנת נתונים בלבד.

## א. שדה שמי A/B — קידוד ידני לפני Python
Train: A,B,A ⇒ לפי סדר group_A/group_B התוצאות [1,0],[0,1],[1,0]. Test: B,C ⇒ [0,1],[0,0]. C חדשה אינה דומה לשתיהן.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
train_group=pd.DataFrame({'group':['A','B','A']})
new_group=pd.DataFrame({'group':['B','C']})
enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False).set_output(transform='pandas')
enc.fit(train_group)
encoded_train=enc.transform(train_group)
encoded_new=enc.transform(new_group)
print('Categories:',enc.categories_)
display(encoded_train,encoded_new)
assert enc.categories_[0].tolist()==['A','B']
assert encoded_train.columns.tolist()==['group_A','group_B']
assert np.array_equal(encoded_train.to_numpy(),[[1,0],[0,1],[1,0]])
assert np.array_equal(encoded_new.to_numpy(),[[0,1],[0,0]])

## ב. רשימת האיים — הדגמת הקורס (seed 42)
הדגמת המורה משתמשת ב־42 וב־island; מטלת התלמיד משתמשת באותו מאפיין אך seed 17. היעד species נשמר בנפרד.

In [ ]:
from sklearn.model_selection import train_test_split
url='https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv'
penguins_df=pd.read_csv(url)
raw=penguins_df.copy(deep=True)
model_df=penguins_df.dropna(subset=['species']).copy()
X=model_df[['island']]
y=model_df['species']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
print('Source',penguins_df.shape,'Train/Test',X_train.shape,X_test.shape)
assert penguins_df.shape==(344,7)
assert X_train.shape==(258,1) and X_test.shape==(86,1)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

## ג. One-Hot על איי האימון בלבד
צפו שלוש עמודות island_Biscoe/island_Dream/island_Torgersen, ו־258/86 שורות ללא שינוי.

In [ ]:
island_encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False).set_output(transform='pandas')
island_encoder.fit(X_train)
X_train_encoded=island_encoder.transform(X_train)
X_test_encoded=island_encoder.transform(X_test)
print('Schema',X_train_encoded.columns.tolist())
display(X_train_encoded.head(3),X_test_encoded.head(3))
assert island_encoder.categories_[0].tolist()==['Biscoe','Dream','Torgersen']
assert X_train_encoded.shape==(258,3)
assert X_test_encoded.shape==(86,3)
assert X_train_encoded.index.equals(y_train.index)
assert X_test_encoded.index.equals(y_test.index)
assert penguins_df.equals(raw)

## ד. שתי הכנות על אותה טבלה — דוגמת מורה
age מספרי, group שמי. חציון age של 10,20 הוא 15; אחרי השלמה mean=15 ו־std=√(50/3), לכן age המתוקנן [−1.2247,+1.2247,0]. Group A/B עובר One-Hot.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
lab_train=pd.DataFrame({'age':[10.,20.,np.nan],'group':['A','B','A']})
lab_new=pd.DataFrame({'age':[15.],'group':['C']})
prepare=ColumnTransformer([
 ('numeric',Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler())]),['age']),
 ('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['group'])
])
train_ready=prepare.fit_transform(lab_train)
new_ready=prepare.transform(lab_new)
print('Feature names:',prepare.get_feature_names_out())
print('Prepared Train:',train_ready)
print('Prepared New:',new_ready)
assert prepare.get_feature_names_out().tolist()==['numeric__age','categorical__group_A','categorical__group_B']
assert train_ready.shape==(3,3) and new_ready.shape==(1,3)
assert np.allclose(train_ready[:,0],[-np.sqrt(1.5),np.sqrt(1.5),0])
assert np.array_equal(train_ready[:,1:],[[1,0],[0,1],[1,0]])
assert np.allclose(new_ready,[[0,0,0]])

## ה. שאלות הגנה
אילו פרמטרים למד המסלול המספרי ומאיזה נתונים? אילו קטגוריות למד המסלול השמי? האם האפסים מוכיחים דמיון? האם נבנה מסווג? **לא**. במעבדה העצמאית יש נתונים אחרים והשלמות קוד.